# DS-11 U.S. Passport Application — Excluded-Class Feature Demo

This notebook demonstrates the **excluded-class feature** — a way to mark
entire document classes as boilerplate (e.g. legal warnings, fee
instructions, tax notices) so the IDP pipeline can **skip them end-to-end**
with zero LLM calls.

## The document

`samples/DS11-USPassportApplication.pdf` is a 6-page US State Department
passport application form with the following structure:

| Page | Content | Nature |
|------|---------|--------|
| 1 | WARNING: False statements… legal notice | Static legal |
| 2 | Passport fee / payment instructions | Static instructions |
| 3 | DS-11 FEDERAL TAX LAW (Section 6039E) notice | Static legal |
| 4 | DS-11 ACTS OR CONDITIONS affidavit | Static oath |
| **5** | **APPLICATION FOR A U.S. PASSPORT (form front)** | **Dynamic form** |
| **6** | **Travel Plans / Permanent Address (form back)** | **Dynamic form** |

Only pages 5–6 carry applicant data. Pages 1–4 are identical across every
DS-11 filed — sending them to an LLM is pure waste.

## What this notebook shows

1. Two new JSON-Schema extensions on the class definition:
   - `x-aws-idp-exclude-from-processing: true`
   - `x-aws-idp-exclusion-reason: instructions`
2. **LLM-based classification** using the multimodal page-level classifier
   (the system default) drives the skip decision via each class's
   `description`. An optional regex fast-path on the excluded class
   short-circuits common boilerplate pages to save tokens, but the LLM
   description is the primary mechanism.
3. Extraction / assessment / summarization / rule validation / evaluation
   all short-circuit on the excluded section and write a consistent
   `{"status": "skipped_excluded_class", ...}` stub.
4. Evaluation metrics stay clean; excluded sections appear in a separate
   **Excluded Sections (Not Evaluated)** table in the markdown report.

See [`docs/classification.md#excluding-static-pages`](../../../docs/classification.md) for the full feature documentation.

## 1. Setup

In [1]:
ROOTDIR = "../../.."
SAMPLE_PDF_PATH = f"{ROOTDIR}/samples/DS11-USPassportApplication.pdf"
CONFIG_PATH = "config/config.yaml"

In [2]:
# Autoreload so edits to idp_common are picked up without restarting the kernel.
%load_ext autoreload
%autoreload 2

import json
import logging
import os
import time
from pathlib import Path

import yaml

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(name)s  %(message)s")
logger = logging.getLogger("ds11-demo")

# Optional: install idp_common from the repo root if not already installed.
# %pip install -q -e "{ROOTDIR}/lib/idp_common_pkg[all]"

### 1.1 Environment — AWS profile, region, auto-named S3 buckets

Uses the `default` boto3 profile (admin-capable — needed to create S3
buckets on the fly). Edit `AWS_PROFILE` below if your admin profile is
named something else. Bucket names are derived from account ID + region;
override with `IDP_INPUT_BUCKET_NAME` / `IDP_OUTPUT_BUCKET_NAME` if needed.

In [3]:
import boto3

# Pick a profile.
AWS_PROFILE = "default"
# Clear any stale per-request AWS credential env vars that would override
# the chosen profile (the shell may have exported AWS_ACCESS_KEY_ID etc.).
for k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"):
    os.environ.pop(k, None)

# Force the default boto3 session to use the chosen profile so that any
# idp_common service which constructs its own boto3 client (OcrService,
# ExtractionService, etc.) picks up the same credentials we do here.
os.environ["AWS_PROFILE"] = AWS_PROFILE
boto3.setup_default_session(profile_name=AWS_PROFILE)
session = boto3.Session(profile_name=AWS_PROFILE)
os.environ["AWS_REGION"] = session.region_name or "us-east-1"
region = os.environ["AWS_REGION"]

# Get AWS account ID for unique bucket names.
sts_client = session.client("sts")
account_id = sts_client.get_caller_identity()["Account"]

# Create unique bucket names based on account ID and region.
input_bucket_name = os.getenv(
    "IDP_INPUT_BUCKET_NAME", f"idp-ds11-demo-input-{account_id}-{region}"
)
output_bucket_name = os.getenv(
    "IDP_OUTPUT_BUCKET_NAME", f"idp-ds11-demo-output-{account_id}-{region}"
)

print(f"Profile       : {AWS_PROFILE}")
print(f"Region        : {region}")
print(f"Account       : {account_id}")
print(f"Input bucket  : {input_bucket_name}")
print(f"Output bucket : {output_bucket_name}")

2026-04-24 20:33:05,545  botocore.credentials  Found credentials in shared credentials file: ~/.aws/credentials


Profile       : default
Region        : us-west-2
Account       : 912625584728
Input bucket  : idp-ds11-demo-input-912625584728-us-west-2
Output bucket : idp-ds11-demo-output-912625584728-us-west-2


### 1.2 Create buckets if they don't exist and upload the DS-11 sample PDF

In [4]:
import datetime
from pathlib import Path

s3_client = session.client("s3")

def ensure_bucket_exists(bucket_name):
    # Only create on a 404 (truly missing). A 403 means the (globally unique)
    # bucket name is already taken — possibly owned by this account in another
    # region or by another principal — so creating would fail; assume it exists.
    from botocore.exceptions import ClientError
    try:
        s3_client.head_bucket(Bucket=bucket_name)
        print(f"Bucket {bucket_name} already exists")
        return
    except ClientError as e:
        code = e.response.get("Error", {}).get("Code", "")
        if code not in ("404", "NoSuchBucket"):
            print(f"Bucket {bucket_name} exists (head_bucket returned {code}); not creating")
            return
    try:
        if region == "us-east-1":
            s3_client.create_bucket(Bucket=bucket_name)
        else:
            s3_client.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={"LocationConstraint": region},
            )
        print(f"Created bucket: {bucket_name}")
        # Deny any non-TLS request to the bucket, matching the EnforceSSLOnly
        # bucket policy the IDP CloudFormation stack applies to its own buckets.
        partition = boto3.Session().get_partition_for_region(region)
        bucket_arn = f"arn:{partition}:s3:::{bucket_name}"
        s3_client.put_bucket_policy(
            Bucket=bucket_name,
            Policy=json.dumps({
                "Version": "2012-10-17",
                "Statement": [{
                    "Sid": "EnforceSSLOnly",
                    "Effect": "Deny",
                    "Principal": "*",
                    "Action": "s3:*",
                    "Resource": [bucket_arn, f"{bucket_arn}/*"],
                    "Condition": {"Bool": {"aws:SecureTransport": "false"}},
                }],
            }),
        )

        s3_client.get_waiter("bucket_exists").wait(Bucket=bucket_name)
    except Exception as e:
        print(f"Error creating bucket {bucket_name}: {e}")
        raise

ensure_bucket_exists(input_bucket_name)
ensure_bucket_exists(output_bucket_name)

# Upload the sample PDF with a timestamped key so reruns don't clobber.
sample_file_key = (
    "ds11-demo-"
    + datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    + Path(SAMPLE_PDF_PATH).suffix
)
with open(SAMPLE_PDF_PATH, "rb") as file_data:
    s3_client.upload_fileobj(file_data, input_bucket_name, sample_file_key)

print(f"Uploaded sample file to: s3://{input_bucket_name}/{sample_file_key}")

Bucket idp-ds11-demo-input-912625584728-us-west-2 already exists
Bucket idp-ds11-demo-output-912625584728-us-west-2 already exists
Uploaded sample file to: s3://idp-ds11-demo-input-912625584728-us-west-2/ds11-demo-2026-04-24_20-33-06.pdf


## 2. Load the config and merge with system defaults

This demo's `config/config.yaml` is a **minimal override config** — it
contains only `notes` + `classes` (no `classification:`, `extraction:`,
`assessment:`, `summarization:`, `ocr:`, or `evaluation:` sections).
All the service-level settings come from the bundled system defaults
(`idp_common/config/system_defaults/pattern-2.yaml` and its inherited
`base-*.yaml` files) via `merge_config_with_defaults()`.

This is exactly the pattern used in production — the
`update_configuration` Lambda performs the same merge when a stack's
`CustomConfigPath` is uploaded.

The merged config uses the **default** classification method
`multimodalPageLevelClassification` — each page is sent (as image +
OCR text) to the LLM and classified independently; consecutive pages
with the same class are then grouped into sections.

In [5]:
from idp_common.config.merge_utils import merge_config_with_defaults

with open(CONFIG_PATH) as f:
    user_config = yaml.safe_load(f)

# Merge minimal-override user config on top of bundled pattern-2 system
# defaults. Result has complete classification/extraction/assessment/
# summarization/ocr/evaluation sections.
CONFIG = merge_config_with_defaults(user_config, pattern="pattern-2")

print(f"User config sections : {sorted(user_config.keys())}")
print(f"Merged config sections: {sorted(CONFIG.keys())}")
print()
print(f"Classification method : {CONFIG['classification']['classificationMethod']}")
print(f"Classification model  : {CONFIG['classification']['model']}")
print(f"Extraction model      : {CONFIG['extraction']['model']}")
print()

classes = CONFIG["classes"]
print(f"Document classes ({len(classes)}):")
for cls in classes:
    name = cls.get("x-aws-idp-document-type") or cls.get("$id")
    excluded = cls.get("x-aws-idp-exclude-from-processing", False)
    reason = cls.get("x-aws-idp-exclusion-reason", "")
    has_regex = bool(cls.get("x-aws-idp-document-page-content-regex"))
    marker = "\U0001f6ab EXCLUDED" if excluded else "\u2705 EXTRACTED"
    extras = []
    if reason:
        extras.append(f"reason={reason!r}")
    if has_regex:
        extras.append("regex-fast-path")
    extras_str = f"  ({', '.join(extras)})" if extras else ""
    print(f"  {marker}  {name}{extras_str}")

User config sections : ['classes', 'notes', 'use_bda']
Merged config sections: ['agents', 'assessment', 'classes', 'classification', 'criteria_validation', 'discovery', 'evaluation', 'extraction', 'notes', 'ocr', 'summarization', 'use_bda']

Classification method : multimodalPageLevelClassification
Classification model  : us.amazon.nova-2-lite-v1:0
Extraction model      : us.anthropic.claude-sonnet-4-6

Document classes (2):
  🚫 EXCLUDED  PassportApplicationInstructions  (reason='instructions', regex-fast-path)
  ✅ EXTRACTED  PassportApplication


## 3. OCR the PDF

We use the `idp_common.ocr` service to split the 6-page PDF into per-page
text and images. OCR is required for both the regex fast-path (text-based)
and the multimodal page-level classifier (which sends text + image).

In [6]:
from idp_common import ocr as idp_ocr
from idp_common.models import Document

doc = Document(
    id="ds11-demo",
    input_bucket=input_bucket_name,
    input_key=sample_file_key,
    output_bucket=output_bucket_name,
)

ocr_service = idp_ocr.OcrService(config=CONFIG)
t0 = time.time()
doc = ocr_service.process_document(doc)
print(f"OCR completed in {time.time() - t0:.1f}s \u2014 {len(doc.pages)} pages parsed.")

from idp_common import s3 as s3util
for pid in sorted(doc.pages.keys(), key=int):
    page = doc.pages[pid]
    try:
        preview = s3util.get_text_content(page.parsed_text_uri)[:120]
    except Exception:
        preview = "<text unavailable>"
    print(f"Page {pid}: {preview!r}...")

2026-04-24 20:33:07,087  botocore.credentials  Found credentials in shared credentials file: ~/.aws/credentials
2026-04-24 20:33:07,194  idp_common.config.models  IDPConfig: Ignoring unknown fields (not defined in model): ['criteria_validation']
2026-04-24 20:33:07,196  idp_common.ocr.service  No image sizing configured, applying default limits: 951x1268 to optimize resource usage and token consumption
2026-04-24 20:33:07,197  idp_common.ocr.service  OCR Service initialized - DPI: None, Image sizing: 951x1268
2026-04-24 20:33:07,197  idp_common.ocr.service  OCR Service initialized with features: ['LAYOUT']
2026-04-24 20:33:07,219  idp_common.ocr.service  OCR Service initialized with Textract backend
2026-04-24 20:33:07,246  idp_common.ocr.service  S3 client initialized with 20 connection pool size
2026-04-24 20:33:07,588  idp_common.ocr.service  Detected file type: pdf
2026-04-24 20:33:07,597  idp_common.ocr.service  Rendering 6 of 6 page images sequentially (pypdfium2 is not thread-sa

OCR completed in 7.0s — 6 pages parsed.
Page 1: '\n\n# U.S. PASSPORT APPLICATION \n\nPLEASE DETACH AND RETAIN THIS INSTRUCTION SHEET FOR YOUR RECORDS \n\n## FOR INFORMATION AN'...
Page 2: '# PROOF OF U.S. CITIZENSHIP \n\nAPPLICANTS BORN IN THE UNITED STATES: Submit a previous U.S. passport or certified birth c'...
Page 3: '## NOTE REGARDING MAILING OF YOUR PASSPORT(S) \n\nPassport Services will not mail a U.S. passport to a private address out'...
Page 4: '## ELECTRONIC PASSPORT STATEMENT \n\nThe U.S. Department of State now issues an "Electronic Passport book, which contains '...
Page 5: '\n\n# APPLICATION FOR A U.S. PASSPORT \n\nPlease Print Legibly Using Black Ink Only \n\nOMB CONTROL NO. 1405-0004 OMB EXPIRATI'...
Page 6: "Name of Applicant (Last, First, & Middle) \n\nDate of Birth (mm/dd/yyyy) \n\n10. Parental Information\nLast Name (at Parent's"...


## 4. Classification — multimodal page-level + optional regex fast-path

The classification service runs two passes per page:

1. **Regex fast-path** (optional): if `x-aws-idp-document-page-content-regex`
   is configured on a class and the page's OCR text matches, the page is
   classified instantly with **zero LLM calls**. Only the excluded class
   has a regex in this demo; it's an optimization for stable boilerplate
   pages.
2. **LLM multimodal classification**: any page that doesn't match a regex
   is sent to the Bedrock model with its image + OCR text. The model
   returns the best-matching class using the `description` fields. This
   is the primary, robust mechanism.

After per-page classification, consecutive pages with the same class are
grouped into sections.

In [7]:
from idp_common.classification.service import ClassificationService

classifier = ClassificationService(config=CONFIG, backend="bedrock")

t0 = time.time()
doc = classifier.classify_document(doc)
print(f"Classification completed in {time.time() - t0:.1f}s")
print()
print(f"Sections ({len(doc.sections)}):")
for sec in doc.sections:
    marker = "\U0001f6ab EXCLUDED" if sec.excluded else "\u2705"
    reason = f" ({sec.exclusion_reason})" if sec.excluded else ""
    print(
        f"  {marker}  section={sec.section_id}  "
        f"class={sec.classification!r}  pages={sec.page_ids}{reason}"
    )

2026-04-24 20:33:14,946  idp_common.config.models  IDPConfig: Ignoring unknown fields (not defined in model): ['criteria_validation']
2026-04-24 20:33:14,948  idp_common.classification.service  Classification caching disabled
2026-04-24 20:33:14,948  idp_common.classification.service  Initialized classification service with Bedrock backend using model us.amazon.nova-2-lite-v1:0
2026-04-24 20:33:14,949  idp_common.classification.service  Using multimodal page-level classification method with document boundary detection
2026-04-24 20:33:14,949  idp_common.classification.service  Classifying document with 6 pages using multimodal page-level classification with bedrock backend
2026-04-24 20:33:14,950  idp_common.classification.service  Attempting to retrieve cached page classifications for document ds11-demo
2026-04-24 20:33:14,951  idp_common.classification.service  Found 0 cached page classifications, classifying 6 remaining pages
2026-04-24 20:33:15,423  idp_common.image  No resize requ

Classification completed in 7.2s

Sections (2):
  🚫 EXCLUDED  section=1  class='PassportApplicationInstructions'  pages=['1', '2', '3', '4'] (instructions)
  ✅  section=2  class='PassportApplication'  pages=['5', '6']


### 4.1 Inspect per-page classifications

Each page also gets a `Page.classification`. Useful for confirming how
each page was classified (regex fast-path vs LLM).

In [8]:
print("Page-level classifications:")
for pid in sorted(doc.pages.keys(), key=int):
    page = doc.pages[pid]
    print(f"  Page {pid}: class={page.classification!r}")

Page-level classifications:
  Page 1: class='PassportApplicationInstructions'
  Page 2: class='PassportApplicationInstructions'
  Page 3: class='PassportApplicationInstructions'
  Page 4: class='PassportApplicationInstructions'
  Page 5: class='PassportApplication'
  Page 6: class='PassportApplication'


## 5. Extraction — excluded sections short-circuit

The `ExtractionService` checks `is_section_excluded(section)` at the top
of `process_document_section`. For the excluded section:

- **No** extraction prompt is built, **no** LLM call is made.
- A stub `result.json` is written containing
  `{"status": "skipped_excluded_class", ...}`.
- `section.extraction_result_uri` points at the stub so downstream stages
  behave exactly like they would for a real section.

The active `PassportApplication` section takes the full normal path and
actually calls the LLM to extract applicant data.

In [9]:
from idp_common.extraction.service import ExtractionService

extractor = ExtractionService(config=CONFIG)

for section in list(doc.sections):
    t0 = time.time()
    doc = extractor.process_document_section(doc, section.section_id)
    elapsed = time.time() - t0
    marker = "\U0001f6ab" if section.excluded else "\u2705"
    print(
        f"{marker}  section={section.section_id}  class={section.classification}  "
        f"elapsed={elapsed:.2f}s  result_uri={section.extraction_result_uri}"
    )

2026-04-24 20:33:26,195  idp_common.config.models  IDPConfig: Ignoring unknown fields (not defined in model): ['criteria_validation']
2026-04-24 20:33:26,196  idp_common.extraction.service  Initialized extraction service with model us.anthropic.claude-sonnet-4-6
2026-04-24 20:33:26,534  idp_common.section_exclusion  Wrote skipped-section stub for extraction stage to s3://idp-ds11-demo-output-912625584728-us-west-2/ds11-demo-2026-04-24_20-33-06.pdf/sections/1/result.json (class=PassportApplicationInstructions, reason=instructions)
2026-04-24 20:33:26,536  idp_common.extraction.service  Extraction skipped for excluded section 1 (class=PassportApplicationInstructions, reason=instructions)
2026-04-24 20:33:26,537  idp_common.extraction.service  Using pre-calculated page_indices from section attributes: [4, 5]
2026-04-24 20:33:26,538  idp_common.extraction.service  Processing 2 pages, class PassportApplication: 5-6


🚫  section=1  class=PassportApplicationInstructions  elapsed=0.34s  result_uri=s3://idp-ds11-demo-output-912625584728-us-west-2/ds11-demo-2026-04-24_20-33-06.pdf/sections/1/result.json


2026-04-24 20:33:27,029  idp_common.extraction.service  Time taken to read text content: 0.35 seconds
2026-04-24 20:33:27,437  idp_common.image  No resize requested (both dimensions are None), returning original image
2026-04-24 20:33:27,593  idp_common.image  No resize requested (both dimensions are None), returning original image
2026-04-24 20:33:27,594  idp_common.extraction.service  Time taken to read images: 0.56 seconds
2026-04-24 20:33:27,594  idp_common.extraction.service  No custom prompt Lambda configured - using default prompt generation
2026-04-24 20:33:27,595  idp_common.extraction.service  Attaching 2 images to extraction prompt
2026-04-24 20:33:27,595  idp_common.image  Detected image format: jpeg
2026-04-24 20:33:27,596  idp_common.image  Detected image format: jpeg
2026-04-24 20:33:27,597  idp_common.extraction.service  Extracting fields for PassportApplication document, section
2026-04-24 20:33:27,666  idp_common.bedrock.client  Processed content with 1 cachepoint ins

✅  section=2  class=PassportApplication  elapsed=9.09s  result_uri=s3://idp-ds11-demo-output-912625584728-us-west-2/ds11-demo-2026-04-24_20-33-06.pdf/sections/2/result.json


## 6. Assessment — excluded sections skipped (no output)

Assessment skips excluded sections entirely. There's no stub to write
because the extraction stub is already authoritative.

In [10]:
from idp_common.assessment.service import AssessmentService

if CONFIG.get("assessment", {}).get("enabled", True):
    assessor = AssessmentService(config=CONFIG)
    for section in doc.sections:
        t0 = time.time()
        doc = assessor.process_document_section(doc, section.section_id)
        elapsed = time.time() - t0
        marker = "\U0001f6ab" if section.excluded else "\u2705"
        print(f"{marker}  assessment section={section.section_id}  elapsed={elapsed:.2f}s")
else:
    print("Assessment disabled in config \u2014 skipping.")

2026-04-24 20:33:35,673  idp_common.config.models  IDPConfig: Ignoring unknown fields (not defined in model): ['criteria_validation']


2026-04-24 20:33:35,675  idp_common.assessment.service  Initialized assessment service with model us.amazon.nova-lite-v1:0
2026-04-24 20:33:35,675  idp_common.assessment.service  Assessment skipped for excluded section 1 (class=PassportApplicationInstructions, reason=instructions)
2026-04-24 20:33:35,676  idp_common.assessment.service  Assessing 2 pages, class PassportApplication: 5-6


🚫  assessment section=1  elapsed=0.00s


2026-04-24 20:33:35,885  idp_common.assessment.service  Time taken to read extraction results: 0.09 seconds
2026-04-24 20:33:36,062  idp_common.assessment.service  Time taken to read text content: 0.18 seconds
2026-04-24 20:33:36,510  idp_common.image  No resize requested (both dimensions are None), returning original image
2026-04-24 20:33:36,653  idp_common.image  No resize requested (both dimensions are None), returning original image
2026-04-24 20:33:36,654  idp_common.assessment.service  Time taken to read images: 0.59 seconds
2026-04-24 20:33:36,822  idp_common.assessment.service  Time taken to read raw OCR results: 0.17 seconds
2026-04-24 20:33:36,823  idp_common.assessment.service  Attaching 2 images to assessment prompt
2026-04-24 20:33:36,824  idp_common.image  Detected image format: jpeg
2026-04-24 20:33:36,825  idp_common.image  Detected image format: jpeg
2026-04-24 20:33:36,826  idp_common.assessment.service  Assessing extraction confidence for PassportApplication documen

✅  assessment section=2  elapsed=10.21s


## 7. Summarization — stub summary for excluded sections

Summarization *does* write a stub (`summary.json`) for excluded sections
so the UI has something to display. The stub has `"stage": "summarization"`.

In [11]:
from idp_common.summarization.service import SummarizationService

if CONFIG.get("summarization", {}).get("enabled", True):
    summarizer = SummarizationService(config=CONFIG)
    for section in doc.sections:
        t0 = time.time()
        doc, _metering = summarizer.process_document_section(doc, section.section_id)
        elapsed = time.time() - t0
        marker = "\U0001f6ab" if section.excluded else "\u2705"
        print(
            f"{marker}  summarization section={section.section_id}  elapsed={elapsed:.2f}s"
        )
else:
    print("Summarization disabled in config \u2014 skipping.")

2026-04-24 20:33:45,910  idp_common.config.models  IDPConfig: Ignoring unknown fields (not defined in model): ['criteria_validation']
2026-04-24 20:33:45,911  idp_common.summarization.service  Initialized summarization service with Bedrock backend using model us.amazon.nova-pro-v1:0


2026-04-24 20:33:46,250  idp_common.section_exclusion  Wrote skipped-section stub for summarization stage to s3://idp-ds11-demo-output-912625584728-us-west-2/ds11-demo-2026-04-24_20-33-06.pdf/sections/1/summary.json (class=PassportApplicationInstructions, reason=instructions)
2026-04-24 20:33:46,253  idp_common.summarization.service  Summarization skipped for excluded section 1 (class=PassportApplicationInstructions, reason=instructions)
2026-04-24 20:33:46,254  idp_common.summarization.service  Summarizing section 2, class PassportApplication: pages 5-6
2026-04-24 20:33:46,344  idp_common.summarization.service  Loaded extraction results for section 2


🚫  summarization section=1  elapsed=0.34s


2026-04-24 20:33:46,522  idp_common.summarization.service  Summarizing text with Bedrock
2026-04-24 20:33:46,590  idp_common.bedrock.client  Bedrock request attempt 1/7:
2026-04-24 20:33:46,591  idp_common.bedrock.client    - model: us.amazon.nova-pro-v1:0
2026-04-24 20:33:46,592  idp_common.bedrock.client    - inferenceConfig: {'temperature': 0.0, 'maxTokens': 4096}
2026-04-24 20:33:46,592  idp_common.bedrock.client    - system: [{'text': "You are a document summarization expert who can analyze and summarize documents from various domains including medical, financial, legal, and general business documents. Your task is to create a summary that captures the key information, main points, and important details from the document. Your output must be in valid JSON format. \\nSummarization Style: Balanced\\\\nCreate a balanced summary that provides a moderate level of detail. Include the main points and key supporting information, while maintaining the document's overall structure. Aim for 

✅  summarization section=2  elapsed=12.63s


## 8. Inspect real vs stub section outputs

Side-by-side view of what the pipeline produced for the **active**
`PassportApplication` section vs. the **stub** written for the excluded
`PassportApplicationInstructions` section:

- **`result.json`** — extraction + assessment (merged). Real fields vs. stub.
- **`summary.json`** — narrative summary vs. stub.
- **`summary.md`** — optional human-readable markdown summary.

In [12]:
from urllib.parse import urlparse

from idp_common import s3 as s3util

def _s3_uri_for(section, filename):
    return (
        f"s3://{doc.output_bucket}/{doc.input_key}/sections/"
        f"{section.section_id}/{filename}"
    )


def _dump(label, payload, max_chars=1500):
    pretty = json.dumps(payload, indent=2, default=str)
    if len(pretty) > max_chars:
        pretty = pretty[:max_chars] + "\n... (truncated)"
    print(f"  {label}:\n{pretty}\n")


for section in doc.sections:
    marker = "\U0001f6ab EXCLUDED" if section.excluded else "\u2705 ACTIVE"
    print(
        f"============================================================\n"
        f"{marker}  section={section.section_id}  class={section.classification}  "
        f"pages={section.page_ids}\n"
        f"============================================================"
    )

    result_uri = section.extraction_result_uri or _s3_uri_for(section, "result.json")
    print(f"\n\U0001f4c4 result.json  ({result_uri})")
    try:
        _dump("extraction+assessment", s3util.get_json_content(result_uri))
    except Exception as e:
        print(f"  (could not read: {e})\n")

    summary_uri = _s3_uri_for(section, "summary.json")
    print(f"\U0001f4dd summary.json  ({summary_uri})")
    try:
        _dump("summary", s3util.get_json_content(summary_uri))
    except Exception as e:
        print(f"  (could not read: {e})\n")

    # Optional markdown summary. Probe with head_object first to avoid
    # noisy ERROR logs in s3util when the object simply doesn't exist.
    summary_md_uri = _s3_uri_for(section, "summary.md")
    _parsed = urlparse(summary_md_uri)
    try:
        s3_client.head_object(Bucket=_parsed.netloc, Key=_parsed.path.lstrip("/"))
        md_body = s3util.get_text_content(summary_md_uri)
        preview = md_body[:600] + ("..." if len(md_body) > 600 else "")
        print(f"\U0001f4dd summary.md  ({summary_md_uri})\n  {preview}\n")
    except s3_client.exceptions.ClientError:
        pass  # summary.md is optional.

🚫 EXCLUDED  section=1  class=PassportApplicationInstructions  pages=['1', '2', '3', '4']

📄 result.json  (s3://idp-ds11-demo-output-912625584728-us-west-2/ds11-demo-2026-04-24_20-33-06.pdf/sections/1/result.json)
  extraction+assessment:
{
  "status": "skipped_excluded_class",
  "stage": "extraction",
  "section_id": "1",
  "classification": "PassportApplicationInstructions",
  "excluded": true,
  "exclusion_reason": "instructions",
  "page_ids": [
    "1",
    "2",
    "3",
    "4"
  ],
  "message": "Section 1 classified as 'PassportApplicationInstructions' which is marked x-aws-idp-exclude-from-processing=true; extraction was skipped.",
  "document_id": "ds11-demo"
}

📝 summary.json  (s3://idp-ds11-demo-output-912625584728-us-west-2/ds11-demo-2026-04-24_20-33-06.pdf/sections/1/summary.json)
  summary:
{
  "status": "skipped_excluded_class",
  "stage": "summarization",
  "section_id": "1",
  "classification": "PassportApplicationInstructions",
  "excluded": true,
  "exclusion_reason":

## 9. Before / after — the ROI

Without the excluded-class flag, the pipeline would classify each page
(fine, that's cheap), but then also run extraction, assessment, and
summarization on every section including the 4 boilerplate pages. With
the flag, the 4 boilerplate pages produce stubs with zero LLM calls.

In [13]:
excluded_sections = [s for s in doc.sections if s.excluded]
active_sections = [s for s in doc.sections if not s.excluded]
excluded_pages = sum(len(s.page_ids) for s in excluded_sections)
active_pages = sum(len(s.page_ids) for s in active_sections)
total_pages = len(doc.pages)

print(f"Total pages              : {total_pages}")
print(f"Active (extracted) pages : {active_pages}")
print(f"Excluded (skipped) pages : {excluded_pages}")
if total_pages:
    saved_pct = 100.0 * excluded_pages / total_pages
    print(f"Pages skipped downstream : ~{saved_pct:.0f}%")
print()
print("Section-level summary:")
for s in doc.sections:
    state = (
        f"\U0001f6ab SKIPPED ({s.exclusion_reason})" if s.excluded
        else "\u2705 EXTRACTED"
    )
    print(f"  section {s.section_id}  pages={s.page_ids}  class={s.classification}  {state}")

Total pages              : 6
Active (extracted) pages : 2
Excluded (skipped) pages : 4
Pages skipped downstream : ~67%

Section-level summary:
  section 1  pages=['1', '2', '3', '4']  class=PassportApplicationInstructions  🚫 SKIPPED (instructions)
  section 2  pages=['5', '6']  class=PassportApplication  ✅ EXTRACTED


---

## See also

- Unit tests: `lib/idp_common_pkg/tests/unit/test_section_exclusion.py`
  (covering Section model, helpers, classification, extraction,
  assessment, summarization, rule_validation, and evaluation)
- Shared helpers: `lib/idp_common_pkg/idp_common/section_exclusion.py`
- Feature documentation: `docs/classification.md#excluding-static-pages-eg-instructions-legal-boilerplate`
- Sample config: `config_library/unified/ds11-passport-application/`